# D1.1 · From alert queue to loop operator

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.0 · Start here — what AI for security operations means](https://spbreed.github.io/cyber-commons/lessons/D1.0.html)**.

| | |
|---|---|
| Tools used | Wazuh, OpenSearch, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Run a triage loop over Wazuh alerts and supervise by exception.

**Why a security engineer needs it.** Supervising by re-reading everything the loop did. The control it builds is: know what the loop must escalate and sample the rest.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The queue does not go away; it changes shape. Instead of triaging alerts you are supervising something that triages alerts, which is a different skill with a different quality bar and a much worse failure mode: confident, fast, and wrong at volume.

> **At CyberTravels.** The analyst on CyberTravels' alerts stops triaging and starts supervising something that triages — which is a different skill, with a worse failure mode: confident, fast, and wrong at volume.

## 2 · The framework

```
   before                          after
   +------------------+            +---------------------------+
   | alert -> analyst |            | alert -> loop -> analyst  |
   |          decides |            |          proposes  reviews|
   +------------------+            +---------------------------+
      100 alerts/day                  1000 alerts/day, 40 reviewed

   new failure mode: confident, fast, and wrong at volume
```

The classic SOC job is a queue: alerts arrive, an analyst reads each one,
decides, and moves on. The constraint is human attention, and it does not scale
— which is why tier-1 burnout and alert fatigue are structural rather than
cultural problems.

The agentic version replaces "read every alert" with "operate a loop that reads
every alert". The analyst's job becomes:

- deciding **what the loop is allowed to conclude** (the verifier, B2.0),
- deciding **what it may do about it** (the tool policy, A3.5),
- and handling the cases it escalates.

The skill that transfers is not triage speed. It is knowing which signals the
loop may believe — because a triage loop with a weak verifier closes true
positives at machine speed, and closing a true positive is silent.

## 3 · Where it breaks — closing a true positive is silent

Every triage decision has two error directions and they are not symmetric. Escalating a false positive costs an analyst ten minutes. **Closing a true positive costs you the incident**, and nothing tells you it happened.

## 4 · The control — the loop may close, but not silently

Three rules make an agentic triage loop safe to run, and none of them is about model quality.

## 5 · The procedure, as a skill

The skill scores a triage loop against ground truth, sweeps the confidence bar so the trade between analyst minutes and missed incidents is made explicitly, and adds the severity floor that no automatic closure may cross whatever its confidence.

In [ ]:
# skills/detection/triage-loop-with-floor/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: triage-loop-with-floor
description: >-
  Run an alert triage loop against ground truth, measure the confusion matrix,
  and bound it with a confidence bar and a severity floor no automatic closure
  may cross. Use when automating triage, or when deciding what an agent may
  close on its own.
allowed-tools: Read, Grep, Glob
---

# The floor is the part that makes the loop deployable

A triage loop that matches ground truth on a sample is a promising loop. What
makes it something you can run is the pair of bounds around it: a confidence bar
that decides when it defers, and a severity floor it may never close through,
whatever its confidence.

## When to use this

Before an agent closes anything, and when tuning how much of a queue is
automated.

## Procedure

**1 — Score against ground truth, as a confusion matrix.** Escalated and closed,
against true and false. Accuracy alone hides the direction of the errors, and
only one direction matters here.

**2 — Sweep the confidence bar.** For each setting, record analyst minutes saved
and incidents missed. This is the trade being made, and it should be made
explicitly by whoever owns the queue rather than implicitly by a default.

**3 — Set the severity floor separately.** Any alert above it is escalated
regardless of confidence. A confident wrong closure on a critical alert is the
failure this exists to prevent, and no confidence threshold protects against it.

**4 — Check what the floor costs.** How many alerts it forces to a human per
day. If that number is above the reading budget, the floor is theatre and the
queue needs a different cut.

**5 — Record every automatic closure with its reason and confidence.** The loop
will be wrong sometimes; the question at review is whether you can find out how.

## Output contract

```json
{
  "confusion": {"tp": 0, "fp": 0, "tn": 0, "fn": 0},
  "confidence_sweep": [{"bar": 0.0, "analyst_minutes_saved": 0, "incidents_missed": 0}],
  "floor": {"severity": "str", "escalated_regardless": 0, "per_day": 0},
  "audit": {"closures_logged": true, "fields": ["reason", "confidence", "rule"]}
}
```

## Failure modes

- **Reporting accuracy.** The direction of the error is the whole question.
- **A floor set above the reading budget.** It routes to a person who will not
  read it.
- **Unlogged closures.** You cannot review what the loop decided.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/detection/triage-loop-with-floor/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/detection/triage-loop-with-floor/scripts/triage_loop_with_floor.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run a triage loop against ground truth, then bound it with a confidence bar and a severity floor that no automatic closure may cross.

This is the executable half of the `triage-loop-with-floor` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
import json, os, urllib.error, urllib.request

# Qwen2.5-7B-Instruct is the floor established in MODELS.md: below it two of
# the lessons' acceptance properties stop holding.
OPEN_WEIGHT_DEFAULT = "qwen2.5-7b-instruct"
TIMEOUT = 60

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        return _openai_compatible(prompt, system, model, max_tokens,
                                  temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the server actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing parameter, and
        # it never contains a key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, serve an open-weight model from")
    print("Kaggle Models and point the adapter at it:")
    print()
    print("   python3 -m llama_cpp.server --model <the .gguf from Kaggle> \\")
    print("           --model_alias qwen2.5-7b-instruct --port 11434 --chat_format qwen")
    print("   export OPENAI_BASE_URL=http://127.0.0.1:11434/v1 \\")
    print("          MODEL=qwen2.5-7b-instruct")
    print()
    print("   MODELS.md has the exact Kaggle download. There is no paid backend:")
    print("   every model result in this repository was produced this way.")


import time
from dataclasses import dataclass, field

@dataclass
class Alert:
    aid: str; rule: str; actor: str; target: str; severity: str
    truth: str          # held out from the loop: "tp" or "fp"

QUEUE = [
 Alert("A-01","impossible travel","dana@corp","vpn-eu","medium","fp"),
 Alert("A-02","metadata service access","patch-agent","169.254.169.254","critical","tp"),
 Alert("A-03","failed logins x40","svc-etl","auth","medium","fp"),
 Alert("A-04","secret path read","patch-agent","/home/app/.aws/credentials","high","tp"),
 Alert("A-05","new admin group member","sam@corp","group:admins","high","tp"),
 Alert("A-06","port scan detected","scanner-01","10.0.0.0/24","low","fp"),
 Alert("A-07","egress to unlisted host","triage-agent","collect.example.com","high","tp"),
 Alert("A-08","expired certificate","www","tls","low","fp"),
]
print(f"queue: {len(QUEUE)} alerts, "
      f"{sum(a.truth=='tp' for a in QUEUE)} true positives")
for a in QUEUE:
    print(f"   {a.aid} {a.severity:8s} {a.rule:24s} {a.actor}")

class ReplayTriage:
    """DETERMINISTIC REPLAY — not a language model. Stands in for a triage model."""
    VERDICTS = {
     "A-01": ("close", 0.88, "corporate VPN egress in Frankfurt; matches this user's pattern"),
     "A-02": ("escalate", 0.97, "link-local metadata endpoint from a non-human identity"),
     "A-03": ("close", 0.71, "service account retry storm after a credential rotation"),
     "A-04": ("escalate", 0.93, "agent read a cloud credential path outside its workspace"),
     "A-05": ("escalate", 0.64, "privileged group change; needs the change ticket checked"),
     "A-06": ("close", 0.90, "authorised internal scanner, scheduled window"),
     "A-07": ("escalate", 0.95, "egress to a host not on the allowlist"),
     "A-08": ("close", 0.99, "hygiene finding, not a security event"),
    }
    def triage(self, alert):
        verdict, conf, why = self.VERDICTS[alert.aid]
        return {"aid": alert.aid, "verdict": verdict, "confidence": conf, "why": why}

model = ReplayTriage()
results = [model.triage(a) for a in QUEUE]
truth = {a.aid: a.truth for a in QUEUE}

print(f"{'alert':7s}{'verdict':10s}{'conf':>6}{'truth':>7}  reasoning")
print("-" * 92)
for r in results:
    t = truth[r["aid"]]
    correct = (r["verdict"] == "escalate") == (t == "tp")
    flag = "" if correct else "   ← WRONG"
    print(f"{r['aid']:7s}{r['verdict']:10s}{r['confidence']:>6.2f}{t:>7}{flag}  {r['why'][:44]}")

def confusion(results, truth):
    tp = fp = tn = fn = 0
    missed = []
    for r in results:
        esc = r["verdict"] == "escalate"
        real = truth[r["aid"]] == "tp"
        if esc and real:      tp += 1
        elif esc and not real: fp += 1
        elif not esc and real: fn += 1; missed.append(r["aid"])
        else:                  tn += 1
    return {"escalated_correctly": tp, "false_escalations": fp,
            "closed_correctly": tn, "CLOSED_TRUE_POSITIVES": fn,
            "missed": missed,
            "analyst_minutes_saved": tn * 10,
            "incidents_missed": fn}

c = confusion(results, truth)
for k, v in c.items(): print(f"{k:26s}{v}")

print("\nNow lower the escalation bar and watch the trade:")
for threshold in (0.5, 0.7, 0.9, 0.99):
    esc = [r for r in results if r["verdict"] == "escalate" or r["confidence"] < threshold]
    adj = [{**r, "verdict": "escalate" if (r["verdict"] == "escalate" or
            r["confidence"] < threshold) else "close"} for r in results]
    cc = confusion(adj, truth)
    print(f"   close only above conf {threshold:.2f} → "
          f"missed {cc['incidents_missed']}, analyst minutes saved "
          f"{cc['analyst_minutes_saved']}")

RULES = {
 "1. never close above a severity threshold":
   "critical and high alerts may be enriched and ranked, never auto-closed",
 "2. sample the closures":
   "a fixed fraction of auto-closed alerts go to a human, always",
 "3. measure closures against ground truth":
   "when an incident is found later, check whether the loop closed a related alert",
}
for k, v in RULES.items(): print(f"{k}\n     {v}")

def safe_triage(alert, verdict, confidence, sample_rate=0.1, seed=0):
    import random, zlib
    # NOT hash(): Python randomises str hashing per process (PYTHONHASHSEED),
    # so hash(alert.aid) picks a different sample on every run and on every
    # machine. crc32 is stable, which is what a sampling rule needs.
    rng = random.Random(zlib.crc32(alert.aid.encode()) % 1000 + seed)
    if verdict == "close" and alert.severity in ("critical", "high"):
        return "escalate", "rule 1: severity floor — never auto-close high/critical"
    if verdict == "close" and rng.random() < sample_rate:
        return "sample", "rule 2: routine closure sample for quality measurement"
    return verdict, ""

print()
adjusted = []
for a in QUEUE:
    r = model.triage(a)
    v, why = safe_triage(a, r["verdict"], r["confidence"])
    adjusted.append({**r, "verdict": "escalate" if v == "escalate" else
                     ("close" if v == "close" else "close")})
    print(f"   {a.aid} {a.severity:8s} {r['verdict']:9s} → {v:9s} {why}")

c2 = confusion(adjusted, truth)
print(f"\nbefore: missed {c['incidents_missed']}   after: missed {c2['incidents_missed']}")
assert c2["incidents_missed"] <= c["incidents_missed"]

# ------------------------------------ the same task, against a real model
# Offline this is a labelled replay; with an open-weight model served
# from Kaggle it is the same code calling a real one.

TASK = 'Triage this alert to one of: escalate, close-benign, needs-context.\n\nAlert: service account svc-reports authenticated from 203.0.113.9 at 03:14 and listed all S3 buckets. svc-reports normally runs hourly from 10.2.0.0/16 and touches one bucket.'

REPLAY = "escalate - the source range and the breadth of the list call are both outside this account's established pattern."

answer, used, model = ask(TASK, replay=REPLAY,
            system='You are a SOC triage assistant. One line: disposition, then why.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("returned one of the three dispositions", any(d in answer.lower() for d in ("escalate", "close-benign", "needs-context")))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## What you just proved

The triage loop escalates 4 alerts and closes 4, matching ground truth on all 8. Lowering the confidence bar trades analyst minutes against missed incidents. The severity floor converts any high or critical closure into an escalation, and the closure sampling routes a fraction of routine closures to a human for quality measurement.

## Your turn

Ask your SOC one question: when an incident is confirmed, does anyone check whether an earlier alert about it was closed? If nobody does, you have no measurement of your false-negative rate — with or without an agent.

---

**Next → [D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*